# nb_housing_build

Builds **one province-year** delivery with the `housing_meta` package and stages it in
the default Lakehouse (`lh_housing`) for the Warehouse procedure `audit.usp_load_run`.

Called by the `pl_housing_load` pipeline. You can also run it by hand: edit the
parameters cell and run all. Generated by `housing-fabric bundle`; regenerate rather than edit.

In [ ]:
# Parameters (the pipeline overrides these)
source_system = "ON"                # Province code, e.g. ON (matches province=XX in Files/raw)
reference_year = 2025               # Reference year (matches year=YYYY in Files/raw)
metadata_release = "current"        # 'current' or a folder name under Files/metadata/releases/
mode = "validate"                   # 'validate' (QA only) or 'publish' (QA, then load the warehouse)
run_id = ""                         # set by the pipeline (@pipeline().RunId)

In [ ]:
# Settings for this workspace
LAKEHOUSE_ROOT = "/lakehouse/default"          # default Lakehouse = lh_housing
TABLES_DIR = f"{LAKEHOUSE_ROOT}/Tables"         # schema-enabled Lakehouse: f"{LAKEHOUSE_ROOT}/Tables/dbo"
WHEEL = f"{LAKEHOUSE_ROOT}/Files/lib/housing_meta-0.2.0-py3-none-any.whl"
KEEP_STAGING_DAYS = 30                         # older staging runs are pruned
WORKSPACE_ID = ""                              # blank = read from the runtime context
LAKEHOUSE_ID = ""                              # blank = read from the runtime context

In [ ]:
%pip install --quiet {WHEEL}

In [ ]:
import json
import uuid

from housing_meta import fabric_api, fabric_run, staging

run_id = run_id or f"manual-{uuid.uuid4()}"
result = fabric_run.run_delivery(
    raw_dir=f"{LAKEHOUSE_ROOT}/Files/raw",
    meta_dir=fabric_run.metadata_dir(f"{LAKEHOUSE_ROOT}/Files", metadata_release),
    tables_dir=TABLES_DIR,
    source_system=source_system,
    reference_year=int(reference_year),
    run_id=run_id,
    metadata_release=metadata_release,
    mode=mode,
    keep_staging_days=KEEP_STAGING_DAYS,
)
print(json.dumps({k: v for k, v in result.items() if k != "qa"}, indent=2))

Refresh the Lakehouse SQL endpoint so the Warehouse procedure sees this run's staging
rows straight away (otherwise metadata sync can lag behind the write).

In [ ]:
import notebookutils

ctx = notebookutils.runtime.context
result["endpoint_refresh"] = fabric_api.refresh_sql_endpoint(
    workspace_id=WORKSPACE_ID or ctx.get("currentWorkspaceId"),
    lakehouse_id=LAKEHOUSE_ID or ctx.get("defaultLakehouseId"),
    tables=staging.STAGING_TABLES,
    token=notebookutils.credentials.getToken("pbi"),
)
print(f"refreshed {len(result['endpoint_refresh'])} staging tables")

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))